<a href="https://colab.research.google.com/github/Alekhya-koda/Anti-Money-Laundering/blob/main/AML_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Downloading the Dataset**

In [2]:
!pip install -q duckdb kagglehub pyarrow
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import kagglehub, os, shutil

handle = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
# If it asks for credentials, run kagglehub.login() first
os.makedirs("/content/data/raw", exist_ok=True)

for f in ["HI-Small_Trans.csv", "HI-Small_accounts.csv"]:
    p = kagglehub.dataset_download(handle, path=f)
    shutil.copy(p, f"/content/data/raw/{f}")
    print("got", f)

!ls -lh /content/data/raw

100%|██████████| 454M/454M [00:02<00:00, 177MB/s]


got HI-Small_Trans.csv


100%|██████████| 32.5M/32.5M [00:00<00:00, 40.3MB/s]


got HI-Small_accounts.csv
total 487M
-rw-r--r-- 1 root root  33M Oct  7 05:59 HI-Small_accounts.csv
-rw-r--r-- 1 root root 454M Oct  7 05:59 HI-Small_Trans.csv


# **Creating the Medallion Architecture**

In [10]:
import os
import duckdb
from types import SimpleNamespace

a = SimpleNamespace(raw="/content/data/raw", variant="HI-Small", out="/content/data")
print(type(a))   # should print SimpleNamespace

BRONZE, SILVER, GOLD = (f"{a.out}/{x}" for x in ("bronze", "silver", "gold"))
for d in (BRONZE, SILVER, GOLD):
    os.makedirs(d, exist_ok=True)

con = duckdb.connect()
con.execute("PRAGMA threads=4")

# ---------------------------------------------------------------- BRONZE
# Raw CSV has TWO columns both called "Account", so we skip the header and
# name the columns ourselves. Everything stays as text/untouched in Bronze.
con.execute(f"""
COPY (
  SELECT * FROM read_csv('{a.raw}/{a.variant}_Trans.csv', header=false, skip=1,
    all_varchar=true,
    names=['ts','from_bank','from_acct','to_bank','to_acct','amt_received',
           'cur_received','amt_paid','cur_paid','pay_format','is_laundering'])
) TO '{BRONZE}/trans.parquet' (FORMAT parquet)
""")

# ---------------------------------------------------------------- SILVER
# Typed, renamed, deduplicated, with simple derived flags. No aggregation yet.
con.execute(f"""
COPY (
  SELECT
    row_number() OVER ()                                   AS txn_id,
    strptime(ts, '%Y/%m/%d %H:%M')                         AS ts,
    from_bank || '_' || from_acct                          AS src_acct,
    to_bank   || '_' || to_acct                            AS dst_acct,
    from_bank, to_bank,
    CAST(amt_paid     AS DOUBLE)                           AS amt_paid,
    CAST(amt_received AS DOUBLE)                           AS amt_received,
    trim(cur_paid)  AS cur_paid,
    trim(cur_received) AS cur_received,
    trim(pay_format) AS pay_format,
    CAST(is_laundering AS INTEGER)                         AS is_laundering,
    (from_bank <> to_bank)                                 AS is_cross_bank,
    (cur_paid <> cur_received)                             AS is_cross_currency,
    (from_bank || from_acct = to_bank || to_acct)          AS is_self_transfer,
    extract(hour FROM strptime(ts, '%Y/%m/%d %H:%M'))      AS hour_of_day
  FROM (SELECT DISTINCT * FROM '{BRONZE}/trans.parquet')
  WHERE amt_paid IS NOT NULL AND ts IS NOT NULL
) TO '{SILVER}/transactions.parquet' (FORMAT parquet)
""")

# ---------------------------------------------------------------- EDA
T = f"'{SILVER}/transactions.parquet'"
def show(title, sql):
    print(f"\n=== {title} ===")
    print(con.execute(sql).df().to_string(index=False))

show("Overview", f"""
  SELECT count(*) AS txns, min(ts) AS first_ts, max(ts) AS last_ts,
         count(DISTINCT src_acct) AS senders, count(DISTINCT dst_acct) AS receivers,
         sum(is_laundering) AS illicit_txns,
         round(100.0*avg(is_laundering),4) AS illicit_pct
  FROM {T}""")

show("Laundering rate by payment format", f"""
  SELECT pay_format, count(*) n, sum(is_laundering) illicit,
         round(100.0*avg(is_laundering),3) illicit_pct
  FROM {T} GROUP BY 1 ORDER BY illicit_pct DESC""")

show("Laundering rate by payment currency (top 10 by volume)", f"""
  SELECT cur_paid, count(*) n, round(100.0*avg(is_laundering),3) illicit_pct
  FROM {T} GROUP BY 1 ORDER BY n DESC LIMIT 10""")

show("Cross-bank / cross-currency / self-transfer", f"""
  SELECT is_cross_bank, is_cross_currency, is_self_transfer, count(*) n,
         round(100.0*avg(is_laundering),3) illicit_pct
  FROM {T} GROUP BY ALL ORDER BY n DESC""")

show("Amount percentiles: all vs illicit (mixed currencies, indicative only)", f"""
  SELECT is_laundering,
         round(quantile_cont(amt_paid,0.5),0)  p50,
         round(quantile_cont(amt_paid,0.9),0)  p90,
         round(quantile_cont(amt_paid,0.99),0) p99,
         round(max(amt_paid),0) max_amt
  FROM {T} GROUP BY 1""")

show("Daily volume and illicit rate", f"""
  SELECT CAST(ts AS DATE) d, count(*) n, sum(is_laundering) illicit
  FROM {T} GROUP BY 1 ORDER BY 1""")

# The business problem's baseline: "flag anything above X". How bad is it?
show("Rule baseline: flag if amt_paid > threshold", f"""
  WITH thr AS (SELECT unnest([1000,10000,50000,100000,500000]) AS x)
  SELECT x AS threshold,
         count(*) FILTER (WHERE amt_paid > x)                               AS alerts,
         round(100.0*sum(is_laundering) FILTER (WHERE amt_paid > x)
               / nullif(count(*) FILTER (WHERE amt_paid > x),0),3)          AS precision_pct,
         round(100.0*sum(is_laundering) FILTER (WHERE amt_paid > x)
               / sum(is_laundering),2)                                      AS recall_pct
  FROM {T}, thr GROUP BY x ORDER BY x""")

# ---------------------------------------------------------------- GOLD
# One row per account: behavioural features + label.
# Label = account SENT at least one laundering transaction.
con.execute(f"""
COPY (
WITH out_f AS (
  SELECT src_acct AS acct,
    count(*) out_cnt, sum(amt_paid) out_sum, avg(amt_paid) out_avg,
    max(amt_paid) out_max, coalesce(stddev(amt_paid),0) out_std,
    count(DISTINCT dst_acct) out_uniq_cp, count(DISTINCT to_bank) out_uniq_banks,
    avg(is_cross_bank::INT) out_cross_bank_share,
    avg(is_cross_currency::INT) out_cross_cur_share,
    avg((pay_format='Cash')::INT) out_cash_share,
    avg((pay_format='Bitcoin')::INT) out_btc_share,
    avg((pay_format='Wire')::INT) out_wire_share,
    avg((hour_of_day < 6)::INT) out_night_share,
    count(DISTINCT CAST(ts AS DATE)) active_days_out,
    max(is_laundering) AS label_sent_illicit
  FROM {T} GROUP BY 1),
in_f AS (
  SELECT dst_acct AS acct,
    count(*) in_cnt, sum(amt_received) in_sum, avg(amt_received) in_avg,
    max(amt_received) in_max,
    count(DISTINCT src_acct) in_uniq_cp, count(DISTINCT from_bank) in_uniq_banks,
    max(is_laundering) AS received_illicit_flag
  FROM {T} GROUP BY 1)
SELECT
  coalesce(o.acct, i.acct) AS acct,
  coalesce(out_cnt,0) out_cnt, coalesce(out_sum,0) out_sum, coalesce(out_avg,0) out_avg,
  coalesce(out_max,0) out_max, coalesce(out_std,0) out_std,
  coalesce(out_uniq_cp,0) out_uniq_cp, coalesce(out_uniq_banks,0) out_uniq_banks,
  coalesce(out_cross_bank_share,0) out_cross_bank_share,
  coalesce(out_cross_cur_share,0) out_cross_cur_share,
  coalesce(out_cash_share,0) out_cash_share, coalesce(out_btc_share,0) out_btc_share,
  coalesce(out_wire_share,0) out_wire_share, coalesce(out_night_share,0) out_night_share,
  coalesce(active_days_out,0) active_days_out,
  coalesce(in_cnt,0) in_cnt, coalesce(in_sum,0) in_sum, coalesce(in_avg,0) in_avg,
  coalesce(in_max,0) in_max, coalesce(in_uniq_cp,0) in_uniq_cp,
  coalesce(in_uniq_banks,0) in_uniq_banks,
  -- flow-through: money in vs out (pass-through accounts are classic layering)
  coalesce(out_sum,0) / nullif(coalesce(in_sum,0),0) AS out_in_ratio,
  coalesce(out_uniq_cp,0) - coalesce(in_uniq_cp,0)   AS fan_diff,
  coalesce(label_sent_illicit,0) AS label
FROM out_f o FULL OUTER JOIN in_f i ON o.acct = i.acct
) TO '{GOLD}/account_features.parquet' (FORMAT parquet)
""")

show("Gold sanity check", f"""
  SELECT count(*) accounts, sum(label) illicit_accounts,
         round(100.0*avg(label),3) illicit_acct_pct
  FROM '{GOLD}/account_features.parquet'""")
print("\nDone. Outputs in", a.out)

<class 'types.SimpleNamespace'>


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Overview ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   txns   first_ts             last_ts  senders  receivers  illicit_txns  illicit_pct
5078336 2022-09-01 2022-09-18 16:18:00   496999     420640        5177.0       0.1019

=== Laundering rate by payment format ===
  pay_format       n  illicit  illicit_pct
         ACH  600793   4483.0        0.746
     Bitcoin  146086     56.0        0.038
        Cash  490891    108.0        0.022
      Cheque 1864331    324.0        0.017
 Credit Card 1323324    206.0        0.016
Reinvestment  481056      0.0        0.000
        Wire  171855      0.0        0.000

=== Laundering rate by payment currency (top 10 by volume) ===
   cur_paid       n  illicit_pct
  US Dollar 1895169        0.101
       Euro 1168296        0.117
Swiss Franc  234860        0.082
       Yuan  213752        0.086
     Shekel  192184        0.049
      Rupee  190202        0.088
   UK Pound  180738        0.073
        Yen  155209        0.100
      Ruble  155178        0.086
    Bitcoin  146061        0.038

=== Cross-ban

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Gold sanity check ===
 accounts  illicit_accounts  illicit_acct_pct
   515088            3376.0             0.655

Done. Outputs in /content/data


In [11]:
!mkdir -p /content/drive/MyDrive/aml_project
!cp -r /content/data/silver /content/data/gold /content/drive/MyDrive/aml_project/

# **EDA**

In [12]:
import duckdb, pandas as pd
con = duckdb.connect()
T = "'/content/data/silver/transactions.parquet'"
G = "'/content/data/gold/account_features.parquet'"

print(con.execute(f"""
SELECT count(*) n,
  count(*) FILTER (WHERE amt_paid <= 0) zero_or_neg,
  count(*) FILTER (WHERE NOT is_cross_currency AND abs(amt_paid-amt_received)>0.01) same_cur_mismatch,
  count(DISTINCT cur_paid) currencies
FROM {T}""").df())

print(con.execute(f"""
SELECT cur_paid, round(quantile_cont(amt_paid,0.5),0) p50, round(max(amt_paid),0) max_amt
FROM {T} GROUP BY 1 ORDER BY max_amt DESC LIMIT 8""").df())

print(con.execute(f"""
SELECT ts >= TIMESTAMP '2022-09-11' AS after_sep10, count(*) n,
       sum(is_laundering) illicit, round(100.0*avg(is_laundering),2) illicit_pct
FROM {T} GROUP BY 1""").df())

         n  zero_or_neg  same_cur_mismatch  currencies
0  5078336            0                  0          15
          cur_paid      p50       max_amt
0              Yen  95950.0  1.046302e+12
1            Rupee  65475.0  9.659333e+11
2            Ruble  70070.0  6.260355e+11
3      Brazil Real   5332.0  1.499632e+11
4     Mexican Peso  18923.0  3.089313e+10
5  Canadian Dollar   1143.0  2.150069e+10
6        US Dollar    904.0  1.662061e+10
7             Euro    747.0  1.346399e+10
   after_sep10        n  illicit  illicit_pct
0        False  5077228   4522.0         0.09
1         True     1108    655.0        59.12


In [13]:
from sklearn.metrics import roc_auc_score
g = con.execute(f"SELECT * FROM {G}").df().fillna(0)
feats = [c for c in g.columns if c not in ("acct", "label")]
auc = pd.Series({f: roc_auc_score(g.label, g[f]) for f in feats})
print(auc.reindex((auc - 0.5).abs().sort_values(ascending=False).index).round(3))

print(g.groupby("label")[["out_cnt","in_cnt","out_uniq_cp","in_uniq_cp",
                          "out_sum","out_cash_share"]].median().T)

out_uniq_cp             0.800
out_uniq_banks          0.800
out_std                 0.790
out_sum                 0.752
out_max                 0.747
out_cnt                 0.734
out_avg                 0.713
out_cross_cur_share     0.708
active_days_out         0.705
in_sum                  0.703
in_max                  0.703
in_avg                  0.686
out_in_ratio            0.676
in_uniq_cp              0.667
in_uniq_banks           0.665
in_cnt                  0.652
fan_diff                0.586
out_cross_bank_share    0.578
out_night_share         0.436
out_cash_share          0.535
out_wire_share          0.527
out_btc_share           0.492
dtype: float64
label                  0          1
out_cnt             2.00       4.00
in_cnt              2.00       8.00
out_uniq_cp         1.00       3.00
in_uniq_cp          2.00       3.00
out_sum         10092.86  299025.67
out_cash_share      0.00       0.00


In [14]:
def p_at_k(col, k):
    top = g.nlargest(k, col)
    return top.label.sum(), round(100*top.label.mean(), 2), round(100*top.label.sum()/g.label.sum(), 1)

print("base rate %:", round(100*g.label.mean(), 3))
for col in ["out_max", "out_sum", "out_cnt", "out_uniq_cp"]:
    for k in (500, 2000, 5000):
        hits, prec, rec = p_at_k(col, k)
        print(f"{col:12s} top {k:5d}: hits={hits:4d} precision={prec:5.2f}% recall={rec:5.1f}%")

base rate %: 0.655
out_max      top   500: hits=  23 precision= 4.60% recall=  0.7%
out_max      top  2000: hits=  68 precision= 3.40% recall=  2.0%
out_max      top  5000: hits= 154 precision= 3.08% recall=  4.6%
out_sum      top   500: hits=  18 precision= 3.60% recall=  0.5%
out_sum      top  2000: hits=  53 precision= 2.65% recall=  1.6%
out_sum      top  5000: hits= 127 precision= 2.54% recall=  3.8%
out_cnt      top   500: hits=  44 precision= 8.80% recall=  1.3%
out_cnt      top  2000: hits=  87 precision= 4.35% recall=  2.6%
out_cnt      top  5000: hits= 139 precision= 2.78% recall=  4.1%
out_uniq_cp  top   500: hits=  37 precision= 7.40% recall=  1.1%
out_uniq_cp  top  2000: hits= 106 precision= 5.30% recall=  3.1%
out_uniq_cp  top  5000: hits= 164 precision= 3.28% recall=  4.9%


In [15]:
import kagglehub, collections
handle = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
pp = kagglehub.dataset_download(handle, path="HI-Small_Patterns.txt")
lines = open(pp).read().splitlines()
begins = [l for l in lines if l.upper().startswith("BEGIN")]
print(len(begins), begins[:3])
print(collections.Counter(l.split(" - ")[1].split(":")[0].strip() for l in begins))

Using Colab cache for faster access to the 'ibm-transactions-for-anti-money-laundering-aml' dataset.
370 ['BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out', 'BEGIN LAUNDERING ATTEMPT - CYCLE:  Max 10 hops', 'BEGIN LAUNDERING ATTEMPT - GATHER-SCATTER:  Max 3-degree Fan-In']
Counter({'CYCLE': 54, 'GATHER-SCATTER': 51, 'BIPARTITE': 49, 'FAN-OUT': 48, 'SCATTER-GATHER': 44, 'STACK': 43, 'RANDOM': 41, 'FAN-IN': 40})


# **Gold Dataset**

In [18]:
import duckdb
con = duckdb.connect()
SILVER = "'/content/data/silver/transactions.parquet'"

# Base table: Sep 1-10 only, alert universe, per-currency amount percentile
con.execute(f"""
CREATE OR REPLACE TABLE tx AS
SELECT ts, src_acct, dst_acct, to_bank, from_bank, pay_format, is_laundering,
       is_cross_bank, amt_paid, cur_paid,
       percent_rank() OVER (PARTITION BY cur_paid ORDER BY amt_paid) AS amt_pct
FROM {SILVER}
WHERE ts < TIMESTAMP '2022-09-11'
  AND NOT is_self_transfer AND NOT is_cross_currency
""")

snaps = [  # name, window_start, cutoff, label_end
    ("train", "2022-09-01", "2022-09-05", "2022-09-07"),
    ("valid", "2022-09-03", "2022-09-07", "2022-09-09"),
    ("test",  "2022-09-05", "2022-09-09", "2022-09-11"),
]

def build(name, ws, c, le):
    return f"""
CREATE OR REPLACE TABLE g_{name} AS
WITH f AS (SELECT * FROM tx WHERE ts >= TIMESTAMP '{ws}' AND ts < TIMESTAMP '{c}'),
o AS (
  SELECT src_acct acct, count(*) out_cnt,
    avg(amt_pct) out_amt_pct_avg, max(amt_pct) out_amt_pct_max,
    count(*) FILTER (WHERE amt_pct > 0.99) out_big_cnt,
    count(DISTINCT dst_acct) out_uniq_cp, count(DISTINCT to_bank) out_uniq_banks,
    avg(is_cross_bank::INT) out_cross_bank_share,
    avg((pay_format='ACH')::INT) out_ach_share,
    avg((pay_format='Cash')::INT) out_cash_share,
    avg((pay_format='Bitcoin')::INT) out_btc_share,
    count(*) FILTER (WHERE ts >= TIMESTAMP '{c}' - INTERVAL 1 DAY) out_cnt_24h,
    count(DISTINCT CAST(ts AS DATE)) out_active_days
  FROM f GROUP BY 1),
i AS (
  SELECT dst_acct acct, count(*) in_cnt,
    avg(amt_pct) in_amt_pct_avg, max(amt_pct) in_amt_pct_max,
    count(DISTINCT src_acct) in_uniq_cp, count(DISTINCT from_bank) in_uniq_banks,
    count(*) FILTER (WHERE ts >= TIMESTAMP '{c}' - INTERVAL 1 DAY) in_cnt_24h
  FROM f GROUP BY 1),
cp AS (
  SELECT src_acct acct, max(n)*1.0/sum(n) out_top_cp_share
  FROM (SELECT src_acct, dst_acct, count(*) n FROM f GROUP BY 1,2) GROUP BY 1),
pairs AS (SELECT DISTINCT src_acct, dst_acct FROM f),
rc AS (
  SELECT a.src_acct acct, count(*) n_reciprocal
  FROM pairs a JOIN pairs b ON a.src_acct = b.dst_acct AND a.dst_acct = b.src_acct
  GROUP BY 1),
seen AS (
  SELECT acct, min(ts) first_ts, max(ts) last_ts FROM
    (SELECT src_acct acct, ts FROM f UNION ALL SELECT dst_acct, ts FROM f)
  GROUP BY 1),
lbl AS (
  SELECT DISTINCT src_acct acct FROM tx
  WHERE is_laundering = 1 AND ts >= TIMESTAMP '{c}' AND ts < TIMESTAMP '{le}')
SELECT '{name}' AS snapshot, s.acct,
  coalesce(out_cnt,0) out_cnt, coalesce(out_amt_pct_avg,0) out_amt_pct_avg,
  coalesce(out_amt_pct_max,0) out_amt_pct_max, coalesce(out_big_cnt,0) out_big_cnt,
  coalesce(out_uniq_cp,0) out_uniq_cp, coalesce(out_uniq_banks,0) out_uniq_banks,
  coalesce(out_cross_bank_share,0) out_cross_bank_share,
  coalesce(out_ach_share,0) out_ach_share, coalesce(out_cash_share,0) out_cash_share,
  coalesce(out_btc_share,0) out_btc_share,
  coalesce(out_cnt_24h,0) out_cnt_24h, coalesce(out_active_days,0) out_active_days,
  coalesce(in_cnt,0) in_cnt, coalesce(in_amt_pct_avg,0) in_amt_pct_avg,
  coalesce(in_amt_pct_max,0) in_amt_pct_max, coalesce(in_uniq_cp,0) in_uniq_cp,
  coalesce(in_uniq_banks,0) in_uniq_banks, coalesce(in_cnt_24h,0) in_cnt_24h,
  coalesce(out_top_cp_share,0) out_top_cp_share,
  coalesce(n_reciprocal,0) n_reciprocal,
  coalesce(out_uniq_cp,0) - coalesce(in_uniq_cp,0) fan_diff,
  coalesce(out_cnt,0) * 1.0 / nullif(coalesce(in_cnt,0),0) out_in_cnt_ratio,
  date_diff('hour', first_ts, last_ts) active_span_hours,
  date_diff('hour', last_ts, TIMESTAMP '{c}') hours_since_last_txn,
  (l.acct IS NOT NULL)::INT AS label
FROM seen s
LEFT JOIN o ON s.acct = o.acct LEFT JOIN i ON s.acct = i.acct
LEFT JOIN cp ON s.acct = cp.acct LEFT JOIN rc ON s.acct = rc.acct
LEFT JOIN lbl l ON s.acct = l.acct
"""

for name, ws, c, le in snaps:
    con.execute(build(name, ws, c, le))

con.execute("""CREATE OR REPLACE TABLE gold_v2 AS
  SELECT * FROM g_train UNION ALL SELECT * FROM g_valid UNION ALL SELECT * FROM g_test""")

print(con.execute("""
  SELECT snapshot, count(*) accounts, sum(label) positives,
         round(100.0*avg(label),3) pos_rate_pct
  FROM gold_v2 GROUP BY 1 ORDER BY 1""").df())

# Coverage: illicit senders in each label window that had NO history in the feature window
for name, ws, c, le in snaps:
    tot = con.execute(f"""SELECT count(DISTINCT src_acct) FROM tx
        WHERE is_laundering=1 AND ts >= TIMESTAMP '{c}' AND ts < TIMESTAMP '{le}'""").fetchone()[0]
    got = con.execute(f"SELECT sum(label) FROM g_{name}").fetchone()[0]
    print(f"{name}: {int(got)}/{tot} illicit senders had prior history ({100*got/tot:.0f}%)")

import os
os.makedirs("/content/drive/MyDrive/aml_project/gold_v2", exist_ok=True)
con.execute("""COPY gold_v2 TO '/content/drive/MyDrive/aml_project/gold_v2/gold_v2.parquet'
               (FORMAT parquet)""")
print("saved")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  snapshot  accounts  positives  pos_rate_pct
0     test    252987      534.0         0.211
1    train    399686      639.0         0.160
2    valid    252878      565.0         0.223
train: 639/751 illicit senders had prior history (85%)
valid: 565/740 illicit senders had prior history (76%)
test: 534/712 illicit senders had prior history (75%)
saved


# **Baselines**

In [19]:
import pandas as pd
g2 = con.execute("SELECT * FROM gold_v2").df().fillna(0)
test = g2[g2.snapshot == "test"]
base = 100 * test["label"].mean()
rows = []
for col in ["out_cnt", "out_uniq_cp", "out_amt_pct_max", "out_big_cnt"]:
    for k in (500, 2000, 5000):
        top = test.nlargest(k, col)
        p = 100 * top["label"].mean()
        rows.append({"baseline": f"v2 rank: {col}, top {k}", "alerts": k,
                     "precision_pct": round(p, 2),
                     "recall_pct": round(100 * top["label"].sum() / test["label"].sum(), 2),
                     "lift_vs_random": round(p / base, 1)})
v2 = pd.DataFrame(rows)
v2.to_csv("/content/drive/MyDrive/aml_project/baselines_v2.csv", index=False)
print("test base rate %:", round(base, 3))
print(v2.to_string(index=False))

test base rate %: 0.211
                          baseline  alerts  precision_pct  recall_pct  lift_vs_random
         v2 rank: out_cnt, top 500     500           4.60        4.31            21.8
        v2 rank: out_cnt, top 2000    2000           1.65        6.18             7.8
        v2 rank: out_cnt, top 5000    5000           0.96        8.99             4.5
     v2 rank: out_uniq_cp, top 500     500           4.00        3.75            19.0
    v2 rank: out_uniq_cp, top 2000    2000           1.95        7.30             9.2
    v2 rank: out_uniq_cp, top 5000    5000           1.04        9.74             4.9
 v2 rank: out_amt_pct_max, top 500     500           2.60        2.43            12.3
v2 rank: out_amt_pct_max, top 2000    2000           1.15        4.31             5.4
v2 rank: out_amt_pct_max, top 5000    5000           0.56        5.24             2.7
     v2 rank: out_big_cnt, top 500     500           3.40        3.18            16.1
    v2 rank: out_big_cnt, top 